# HStream Extractor (Colab)

Requires **hanime-plugin**.
Subtitles: tries **old host** first, then **imoto-str.ane-h.xyz**.
Fill XSRF-TOKEN + hstream_session. Optional SERIES_SLUG (dots).

In [ ]:
import os
import subprocess
import requests
import glob
from tqdm.notebook import tqdm

print("Installing dependencies...")
try:
    subprocess.run(["pip", "install", "--upgrade", "yt-dlp", "requests", "tqdm", "hanime-plugin"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "ffmpeg"], check=True)
    if subprocess.run(["which", "deno"], capture_output=True).returncode != 0:
        print("Installing deno...")
        subprocess.run("curl -fsSL https://deno.land/install.sh | sh", shell=True, check=False)
        os.environ["PATH"] = os.path.expanduser("~/.deno/bin") + os.pathsep + os.environ.get("PATH", "")
    print("Dependencies installed successfully!")
except Exception as e:
    print(f"Warning during dependency setup: {e}")

In [ ]:
# @title Settings
URL_LIST = "https://hstream.moe/hentai/houkago-nureta-seifuku-2"  #@param {type:"string"}
DESTINATION_FOLDER = "/content/downloads"  #@param {type:"string"}
XSRF_TOKEN = ""  #@param {type:"string"}
HSTREAM_SESSION = ""  #@param {type:"string"}
SERIES_SLUG = ""  #@param {type:"string"}
MAKE_SAMPLE = True  #@param {type:"boolean"}
SAMPLE_START = "00:12:01"  #@param {type:"string"}
SAMPLE_DURATION_SEC = 60  #@param {type:"integer"}

print("Settings loaded")
print("SERIES_SLUG:", SERIES_SLUG.strip() or "(auto)")
print("Sample:", MAKE_SAMPLE, SAMPLE_START, f"{SAMPLE_DURATION_SEC}s")

In [ ]:
if not os.path.exists(DESTINATION_FOLDER):
    os.makedirs(DESTINATION_FOLDER)

deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.isdir(deno_bin) and deno_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")

urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"Found {len(urls)} links to process.\n")

cookie_parts = []
if XSRF_TOKEN.strip():
    cookie_parts.append(f"XSRF-TOKEN={XSRF_TOKEN.strip()}")
if HSTREAM_SESSION.strip():
    cookie_parts.append(f"hstream_session={HSTREAM_SESSION.strip()}")
COOKIE_HEADER = "; ".join(cookie_parts)

if not COOKIE_HEADER:
    print("WARNING: No cookies set.\n")

# Old host first, then new host
SUB_HOSTS = [
    "https://oppai-str.shoujo-h.org",
    "https://imoto-str.ane-h.xyz",
]

def try_download_sub(sub_url, sub_path):
    try:
        r = requests.get(sub_url, stream=True, timeout=30)
        if r.status_code != 200:
            return False
        total = int(r.headers.get("content-length", 0))
        with open(sub_path, "wb") as f, tqdm(
            desc="Subtitle Progress", total=total, unit="B",
            unit_scale=True, unit_divisor=1024, leave=False,
        ) as bar:
            for chunk in r.iter_content(1024):
                bar.update(len(chunk))
                f.write(chunk)
        return True
    except Exception:
        return False

for index, url in enumerate(tqdm(urls, desc="Overall Progress", unit="video"), start=1):
    tqdm.write(f"\nProcessing [{index}/{len(urls)}]: {url}")

    output_template = os.path.join(DESTINATION_FOLDER, "%(title)s.%(ext)s")
    cmd = ["yt-dlp", "-v", "--downloader", "aria2c", "--concurrent-fragments", "8", "-o", output_template, "--no-mtime"]
    if COOKIE_HEADER:
        cmd += ["--add-header", f"Cookie: {COOKIE_HEADER}"]
    cmd.append(url)

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        tqdm.write(f"Error downloading: {url}")
        err = (e.stderr or e.stdout or str(e))
        for line in err.strip().splitlines()[-40:]:
            tqdm.write(line)
        tqdm.write("Retrying with --downloader ffmpeg ...")
        cmd2 = ["yt-dlp", "--downloader", "ffmpeg", "-o", output_template, "--no-mtime"]
        if COOKIE_HEADER:
            cmd2 += ["--add-header", f"Cookie: {COOKIE_HEADER}"]
        cmd2.append(url)
        try:
            subprocess.run(cmd2, check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e2:
            for line in (e2.stderr or e2.stdout or str(e2)).strip().splitlines()[-20:]:
                tqdm.write(line)
            continue

    files = [f for f in glob.glob(os.path.join(DESTINATION_FOLDER, "*"))
             if not f.endswith(".ass") and "-sample" not in f.lower()]
    if not files:
        tqdm.write("No output files found. Skipping...")
        continue

    latest_video = max(files, key=os.path.getctime)
    base_name = os.path.splitext(os.path.basename(latest_video))[0]
    final_mkv = os.path.join(DESTINATION_FOLDER, f"{base_name}.mkv")

    if latest_video.endswith(".mkv"):
        tqdm.write(f"Already MKV: {latest_video}")
        continue

    parts = url.rstrip("/").split("/")
    ep_token = parts[-1]
    ep_num = ep_token.split("-")[-1]
    url_slug = "-".join(ep_token.split("-")[:-1]) if "-" in ep_token else ep_token

    candidates = []
    if SERIES_SLUG.strip():
        candidates.append(SERIES_SLUG.strip())
    candidates += [
        url_slug.replace("-", "."),
        url_slug,
        ".".join(w.capitalize() for w in url_slug.split("-")),
    ]
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]

    sub_path = os.path.join(DESTINATION_FOLDER, f"{base_name}.ass")
    tqdm.write("Downloading subtitle...")
    sub_ok = False
    for host in SUB_HOSTS:
        for slug in candidates:
            sub_url = f"{host}/2024/{slug}/E{int(ep_num):02d}/eng.ass"
            tqdm.write(f"  try: {sub_url}")
            if try_download_sub(sub_url, sub_path):
                sub_ok = True
                tqdm.write(f"  found on {host}")
                break
        if sub_ok:
            break

    if sub_ok:
        try:
            tqdm.write("Remuxing video and subtitles into MKV...")
            subprocess.run([
                "ffmpeg", "-y", "-i", latest_video, "-i", sub_path,
                "-map", "0", "-map", "1", "-c", "copy",
                "-metadata:s:s:0", "language=eng", final_mkv
            ], check=True)
            if os.path.exists(sub_path):
                os.remove(sub_path)
            if latest_video != final_mkv and os.path.exists(latest_video):
                os.remove(latest_video)
            tqdm.write(f"Successfully Saved: {final_mkv}")
        except Exception as ex:
            tqdm.write(f"Error remux: {ex}")
    else:
        tqdm.write("Subtitle not found on old or new host. Keeping original video.")
        tqdm.write("Tip: set SERIES_SLUG e.g. Houkago.Nureta.Seifuku")

print("\n" + "=" * 50)
print("ALL TASKS COMPLETED!")

In [ ]:
# @title Make 1-minute samples from finished videos
import re
from pathlib import Path

def parse_ts(ts):
    parts = [int(x) for x in ts.strip().split(":")]
    if len(parts) == 3: return parts[0]*3600 + parts[1]*60 + parts[2]
    if len(parts) == 2: return parts[0]*60 + parts[1]
    return parts[0]

def fmt_ts(total):
    h, r = divmod(max(0, total), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def safe_name(name):
    name = re.sub(r'[\\/:*?"<>|]', '', name)
    return re.sub(r"\s+", " ", name).strip()

dest = Path(DESTINATION_FOLDER)
start_sec = parse_ts(SAMPLE_START)
dur = int(SAMPLE_DURATION_SEC)
start_label, end_label = fmt_ts(start_sec), fmt_ts(start_sec + dur)
mins = max(1, int(round(dur / 60)))

videos = sorted([p for p in dest.iterdir() if p.suffix.lower() in {".mkv",".mp4",".webm",".ts"} and "-sample" not in p.stem.lower()])

if not MAKE_SAMPLE:
    print("MAKE_SAMPLE is False – skipping.")
elif not videos:
    print("No videos found in", dest)
else:
    print(f"Creating {dur}s samples from {start_label} -> {len(videos)} file(s)\n")
    for vid in videos:
        out_name = f"{safe_name(vid.stem)}-sample [{start_label} - {end_label}] {mins} Minute{vid.suffix}"
        out_path = dest / out_name
        print(f"Sample: {vid.name} -> {out_name}")
        cmd = ["ffmpeg", "-y", "-ss", str(start_sec), "-i", str(vid), "-t", str(dur), "-c", "copy", str(out_path)]
        try:
            subprocess.run(cmd, check=True, capture_output=True)
            print(f"  OK: {out_path}")
        except subprocess.CalledProcessError:
            cmd2 = ["ffmpeg", "-y", "-ss", str(start_sec), "-i", str(vid), "-t", str(dur),
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-c:a", "aac", "-b:a", "128k", str(out_path)]
            try:
                subprocess.run(cmd2, check=True, capture_output=True)
                print(f"  OK (re-encode): {out_path}")
            except subprocess.CalledProcessError as e2:
                print(f"  FAILED: {e2}")
    print("\nSamples done.")

In [ ]:
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print("Created: /content/hstream_downloads.zip")